# 02 - Smarter filling, new features, tree models

Changes from notebook 01:
1. **Fill gaps with logic** before using averages (e.g. a sleeper spent 0).
2. **New features:** cabin deck/number/side, total spend, group size, family size, child flag.
3. **Tree models:** HistGradientBoosting, LightGBM, XGBoost, and a soft-voting ensemble.

Output: `submission_v2.csv`.

In [1]:
# --- Libraries ---
import warnings
warnings.filterwarnings("ignore")                     # hide harmless library warnings
import numpy as np                                    # maths helpers
import pandas as pd                                   # tables (DataFrames)
from sklearn.linear_model import LogisticRegression   # a simple yes/no model
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score   # tools to check accuracy
from sklearn.pipeline import make_pipeline            # chain steps: scale data -> model
from sklearn.preprocessing import StandardScaler      # puts numbers on the same scale

# --- Load the data ---
train = pd.read_csv("spaceship-titanic-data/train.csv")   # has the answer column "Transported"
test = pd.read_csv("spaceship-titanic-data/test.csv")     # no answers - we predict these

# Stack train and test into one table, so cleaning steps are applied the same way to both
df = pd.concat([train, test], ignore_index=True)
is_train = df.Transported.notna()                     # True for train rows, False for test rows

# The five money columns (how much each passenger spent on board)
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# PassengerId looks like "0013_01": first 4 digits = travel group, last 2 = person in that group
df["Group"] = df.PassengerId.str[:4]

y = train.Transported.astype(int)                     # the answers as 1 (transported) / 0 (not)
groups = df.loc[is_train, "Group"]                    # group of each train passenger

# How we check accuracy: split train into 5 parts, train on 4, test on the 5th, repeat 5 times.
# "Group" version keeps a whole travel group in the same part. Otherwise the model could
# learn from someone's family and be tested on them, which makes the score look too good.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

# --- Extra models used in this notebook ---
from lightgbm import LGBMClassifier                   # fast tree model (gradient boosting)
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier                     # another popular tree model

## 1. Fill gaps with logic
Filling uses train + test together. That's safe: it never uses the `Transported` answers.

In [2]:
# Remember how many gaps there were, to compare at the end
before = df.isna().sum()

# Rule 1: sleepers (CryoSleep = True) can't spend money -> their missing spending is 0
sleeping = df.CryoSleep == True
df.loc[sleeping, SPEND] = df.loc[sleeping, SPEND].fillna(0)

# Rule 2: anyone who spent money was awake -> CryoSleep = False
spent = df[SPEND].sum(axis=1, min_count=1)       # min_count=1: stays empty if all 5 are empty
df.loc[df.CryoSleep.isna() & (spent > 0), "CryoSleep"] = False

# Rule 3: people in the same group (or with the same surname) come from the same planet.
# For each group, find its most common planet, then use it to fill the gaps
df["Surname"] = df.Name.str.split().str[-1]      # last word of the name
for key in ["Group", "Surname"]:
    planet = df.dropna(subset=["HomePlanet"]).groupby(key).HomePlanet.agg(lambda s: s.mode()[0])
    df["HomePlanet"] = df.HomePlanet.fillna(df[key].map(planet))

# Rule 4: split Cabin "B/0/P" into Deck, CabinNum, Side.
# Group members usually share a deck and side, so fill gaps from the group
cabin = df.Cabin.str.split("/", expand=True)
df["Deck"], df["CabinNum"], df["Side"] = cabin[0], pd.to_numeric(cabin[1]), cabin[2]
for c in ["Deck", "Side"]:
    common = df.dropna(subset=[c]).groupby("Group")[c].agg(lambda s: s.mode()[0])
    df[c] = df[c].fillna(df.Group.map(common))

# Show how many gaps the rules filled (Deck and Side started with the same gaps as Cabin)
after = df.isna().sum()
before["Deck"] = before["Side"] = before["Cabin"]
pd.DataFrame({"Missing before": before, "Missing after logic": after}).loc[
    ["CryoSleep", "HomePlanet", "RoomService", "Spa", "Deck", "Side"]].astype(int)

,Missing before,Missing after logic
CryoSleep,310,136
HomePlanet,288,13
RoomService,263,170
Spa,284,177
Deck,299,162
Side,299,162


### Fill what's left with simple values

In [3]:
# Money still missing -> 0 (most passengers spend nothing on most things)
for c in SPEND:
    df[c] = df[c].fillna(0)

# CryoSleep still missing -> guess "asleep" if they spent nothing at all
df["CryoSleep"] = df.CryoSleep.fillna(df[SPEND].sum(axis=1) == 0)

# Age -> median age from train. Cabin number -> median cabin number
df["Age"] = df.Age.fillna(df[is_train].Age.median())
df["CabinNum"] = df.CabinNum.fillna(df.CabinNum.median())

# Text columns -> the most common value (the "mode")
for c in ["HomePlanet", "Destination", "Deck", "Side"]:
    df[c] = df[c].fillna(df[c].mode()[0])
df["VIP"] = df.VIP.fillna(False)

## 2. New features

In [4]:
# --- New columns built from existing ones ---
df["TotalSpend"] = df[SPEND].sum(axis=1)                              # total money spent
df["NoSpend"] = (df.TotalSpend == 0).astype(int)                      # 1 if spent nothing
df["GroupSize"] = df.groupby("Group").Group.transform("size")         # people in the travel group
df["Alone"] = (df.GroupSize == 1).astype(int)                         # 1 if travelling alone
df["FamilySize"] = df.groupby("Surname").Surname.transform("size").fillna(1)   # people with same surname
df["CabinRegion"] = (df.CabinNum // 300).astype(int)                  # cabins in blocks of 300
df["Child"] = (df.Age < 13).astype(int)                               # 1 if under 13

# --- Build the final table of numbers for the model ---
X_all = pd.DataFrame(index=df.index)

# Money: log1p shrinks big amounts (e.g. 10,000 -> 9.2) so a few big spenders don't dominate
for c in SPEND + ["TotalSpend"]:
    X_all[c] = np.log1p(df[c])

# Number columns used as they are
for c in ["Age", "NoSpend", "GroupSize", "Alone", "FamilySize", "CabinNum", "CabinRegion", "Child"]:
    X_all[c] = df[c]

# Yes/no columns -> 1/0
X_all["CryoSleep"] = df.CryoSleep.astype(int)
X_all["VIP"] = df.VIP.astype(int)

# Text columns -> one 0/1 column per value
for c in ["HomePlanet", "Destination", "Deck", "Side"]:
    X_all = X_all.join(pd.get_dummies(df[c], prefix=c).astype(int))

# Split back into train and test
X, X_test = X_all[is_train], X_all[~is_train]
print(X.shape[1], "features")
X.head()

32 features


,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,TotalSpend,Age,NoSpend,GroupSize,Alone,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_P,Side_S
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,39.0,1,1,1,...,0,1,0,0,0,0,0,0,1,0
1,4.700480,2.302585,3.258097,6.309918,3.806662,6.602588,24.0,0,1,1,...,0,0,0,0,0,1,0,0,0,1
2,3.784190,8.182280,0.000000,8.812248,3.912023,9.248021,58.0,0,2,0,...,1,0,0,0,0,0,0,0,0,1
3,0.000000,7.157735,5.918894,8.110728,5.267858,8.551981,33.0,0,2,0,...,1,0,0,0,0,0,0,0,0,1
4,5.717028,4.262680,5.023881,6.338594,1.098612,6.995766,16.0,0,1,1,...,0,0,0,0,0,1,0,0,0,1


## 3. Compare models

In [5]:
# The models to compare. The settings (n_estimators, learning_rate ...) are common safe choices:
#   n_estimators  = how many small trees to build
#   learning_rate = how much each new tree corrects the previous ones (small = careful)
#   max_depth / num_leaves = how complex each tree can be
#   subsample / colsample_bytree = each tree sees a random part of rows / columns (reduces overfitting)
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=20, random_state=0),
    "LightGBM": LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=20, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.7, random_state=0, verbose=-1),
    "XGBoost": XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=5, subsample=0.8,
                             colsample_bytree=0.7, random_state=0, verbosity=0),
}

# Ensemble: average the predicted chances of the 3 tree models ("soft" voting)
models["Voting (3 tree models)"] = VotingClassifier(
    [(n, models[n]) for n in ["HistGradientBoosting", "LightGBM", "XGBoost"]], voting="soft")

# Check each model with the same 5-part group split, and show a table
results = {}
for name, m in models.items():
    s = cross_val_score(m, X, y, cv=cv, groups=groups)
    results[name] = {"Accuracy": s.mean(), "+/-": s.std()}   # +/- = how much it varies between parts
pd.DataFrame(results).T.round(3)

,Accuracy,+/-
Logistic regression,0.776,0.006
HistGradientBoosting,0.809,0.002
LightGBM,0.811,0.005
XGBoost,0.810,0.005
Voting (3 tree models),0.811,0.004


**Tree models beat logistic regression by about 2.5%.**
LightGBM, XGBoost and the vote are within a fraction of a percent of each other.
The vote is used: averaging several models is usually steadier on new data.

## 4. Which features matter most? (LightGBM)

In [6]:
# Train LightGBM on all train data, then see which columns it used most to split the data
lgbm = models["LightGBM"].fit(X, y)
pd.Series(lgbm.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)

CabinNum        1361
TotalSpend       960
Age              911
VRDeck           820
Spa              781
FoodCourt        738
ShoppingMall     646
RoomService      634
FamilySize       484
Side_P           291
GroupSize        213
CabinRegion      153
dtype: int32

## 5. Write submission v2

In [7]:
# Train the voting ensemble on ALL train data, then predict the test passengers
final = models["Voting (3 tree models)"].fit(X, y)

# Kaggle wants True/False, so convert 1/0 with astype(bool)
sub = pd.DataFrame({"PassengerId": test.PassengerId, "Transported": final.predict(X_test).astype(bool)})
sub.to_csv("submission_v2.csv", index=False)

# Compare with v1: how many passengers got a different answer?
v1 = pd.read_csv("submission_v1.csv")
print(len(sub), "rows,", f"{sub.Transported.mean():.0%} predicted transported")
print("Different from v1:", (sub.Transported.values != v1.Transported.values).sum(), "passengers")

4277 rows, 51% predicted transported
Different from v1: 542 passengers


## Summary
| Version | Check on train |
|---|---|
| v1 logistic regression, simple fills | ~0.786 |
| **v2 logic fills + features + tree vote** | **~0.81** |

- On this competition, honest scores are usually **0.80–0.81**. Very few go above 0.82.
- Unlike Titanic, this dataset is large (8,693 rows), so the check on train should be closer to the Kaggle score.